In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

import joblib

In [9]:
df = pd.read_csv("Breast_Cure.csv")

print(df.head())

print("\nShape :", df.shape)

print("\nColumns :")
print(df.columns)

         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  texture_worst  perimeter_worst  area_worst  smoothness

In [10]:
df.drop(["id"], axis=1, inplace=True)


In [11]:
selected_features = [
    "radius_mean",
    "texture_mean",
    "perimeter_mean",
    "area_mean",
    "concavity_mean",
    "concave points_mean",
    "diagnosis"
]

df = df[selected_features]

print(df.head())
print(df.shape)

   radius_mean  texture_mean  perimeter_mean  area_mean  concavity_mean  \
0        17.99         10.38          122.80     1001.0          0.3001   
1        20.57         17.77          132.90     1326.0          0.0869   
2        19.69         21.25          130.00     1203.0          0.1974   
3        11.42         20.38           77.58      386.1          0.2414   
4        20.29         14.34          135.10     1297.0          0.1980   

   concave points_mean diagnosis  
0              0.14710         M  
1              0.07017         M  
2              0.12790         M  
3              0.10520         M  
4              0.10430         M  
(569, 7)


In [12]:
df["diagnosis"] = df["diagnosis"].map({
    "M": 1,
    "B": 0
})

print(df.head())

   radius_mean  texture_mean  perimeter_mean  area_mean  concavity_mean  \
0        17.99         10.38          122.80     1001.0          0.3001   
1        20.57         17.77          132.90     1326.0          0.0869   
2        19.69         21.25          130.00     1203.0          0.1974   
3        11.42         20.38           77.58      386.1          0.2414   
4        20.29         14.34          135.10     1297.0          0.1980   

   concave points_mean  diagnosis  
0              0.14710          1  
1              0.07017          1  
2              0.12790          1  
3              0.10520          1  
4              0.10430          1  


In [13]:
X = df.drop("diagnosis", axis=1)

y = df["diagnosis"]

print("X Shape :", X.shape)
print("y Shape :", y.shape)

X Shape : (569, 6)
y Shape : (569,)


In [15]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.33,
    random_state=42
)

# Different K values
k_values = [3, 5, 7, 11, 13, 15]

best_accuracy = 0
best_k = 0

for k in k_values:

    modelKNN = KNeighborsClassifier(n_neighbors=k)

    modelKNN.fit(X_train, y_train)

    y_pred = modelKNN.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    print(f"K = {k} --> Accuracy = {accuracy}")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_k = k

print("\nBest K Value :", best_k)
print("Best Accuracy :", round(best_accuracy, 4))

K = 3 --> Accuracy = 0.9095744680851063
K = 5 --> Accuracy = 0.9042553191489362
K = 7 --> Accuracy = 0.9095744680851063
K = 11 --> Accuracy = 0.9148936170212766
K = 13 --> Accuracy = 0.9202127659574468
K = 15 --> Accuracy = 0.925531914893617

Best K Value : 15
Best Accuracy : 0.9255


In [17]:
C_values = [1, 10, 20]
kernels = ["linear", "rbf"]

best_accuracy = 0
best_C = 0
best_kernel = ""

for c in C_values:

    for kernel in kernels:

        modelSVM = SVC(
            C=c,
            kernel=kernel,
            random_state=42
        )

        modelSVM.fit(X_train, y_train)

        y_pred = modelSVM.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)

        print(f"C = {c} | Kernel = {kernel} --> Accuracy = {accuracy}")

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_C = c
            best_kernel = kernel

print("\nBest C :", best_C)
print("Best Kernel :", best_kernel)
print("Best Accuracy :", round(best_accuracy, 4))

C = 1 | Kernel = linear --> Accuracy = 0.9414893617021277
C = 1 | Kernel = rbf --> Accuracy = 0.8936170212765957
C = 10 | Kernel = linear --> Accuracy = 0.9202127659574468
C = 10 | Kernel = rbf --> Accuracy = 0.9202127659574468
C = 20 | Kernel = linear --> Accuracy = 0.9414893617021277
C = 20 | Kernel = rbf --> Accuracy = 0.9202127659574468

Best C : 1
Best Kernel : linear
Best Accuracy : 0.9415


In [ ]:
param_grid = {
    "C": [1, 10, 20],
    "kernel": ["linear", "rbf"]
}

grid_search = GridSearchCV(
    estimator=SVC(),
    param_grid=param_grid,
    cv=5
)

grid_search.fit(X_train, y_train)

grid_results = pd.DataFrame(grid_search.cv_results_)

print(
    grid_results[
        [
            "param_C",
            "param_kernel",
            "mean_test_score"
        ]
    ]
)

print("\nBest Parameters :", grid_search.best_params_)
print("Best Score :", round(grid_search.best_score_, 4))

   param_C param_kernel  mean_test_score
0        1       linear         0.897608
1        1          rbf         0.866200
2       10       linear         0.900205
3       10          rbf         0.871429
4       20       linear         0.910731
5       20          rbf         0.874060

Best Parameters : {'C': 20, 'kernel': 'linear'}
Best Score : 0.9107


In [20]:
random_search = RandomizedSearchCV(
    estimator=SVC(),
    param_distributions=param_grid,
    n_iter=5,
    cv=5,
    random_state=42
)

random_search.fit(X_train, y_train)

random_results = pd.DataFrame(
    random_search.cv_results_
)

print(
    random_results[
        [
            "param_C",
            "param_kernel",
            "mean_test_score"
        ]
    ]
)

print("\nBest Parameters :", random_search.best_params_)
print("Best Score :", round(random_search.best_score_, 4))

   param_C param_kernel  mean_test_score
0        1       linear         0.897608
1        1          rbf         0.866200
2       20          rbf         0.874060
3       10       linear         0.900205
4       20       linear         0.910731

Best Parameters : {'kernel': 'linear', 'C': 20}
Best Score : 0.9107


In [21]:
from sklearn.ensemble import RandomForestClassifier

X_train_RF, X_test_RF, y_train_RF, y_test_RF = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

modelRF = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

modelRF.fit(X_train_RF, y_train_RF)

y_pred_RF = modelRF.predict(X_test_RF)

accuracy_RF = accuracy_score(y_test_RF, y_pred_RF)

print("Random Forest Accuracy :", round(accuracy_RF, 4))

Random Forest Accuracy : 0.9561


In [22]:
# AdaBoost
modelAda = AdaBoostClassifier(
    n_estimators=100,
    random_state=42
)

modelAda.fit(X_train, y_train)

y_pred_Ada = modelAda.predict(X_test)

accuracy_Ada = accuracy_score(y_test, y_pred_Ada)

print("AdaBoost Accuracy :", round(accuracy_Ada, 4))


# Gradient Boosting
modelGB = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

modelGB.fit(X_train, y_train)

y_pred_GB = modelGB.predict(X_test)

accuracy_GB = accuracy_score(y_test, y_pred_GB)

print("Gradient Boosting Accuracy :", round(accuracy_GB, 4))

AdaBoost Accuracy : 0.9415
Gradient Boosting Accuracy : 0.9521


In [24]:
modelXGB = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

modelXGB.fit(X_train, y_train)

y_pred_XGB = modelXGB.predict(X_test)

accuracy_XGB = accuracy_score(y_test, y_pred_XGB)

print("XGBoost Accuracy :", round(accuracy_XGB, 4))

XGBoost Accuracy : 0.9734


In [25]:
modelXGB = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

modelXGB.fit(X_train, y_train)

y_pred_XGB = modelXGB.predict(X_test)

accuracy_XGB = accuracy_score(y_test, y_pred_XGB)

print("XGBoost Accuracy :", round(accuracy_XGB, 4))

XGBoost Accuracy : 0.9734


In [26]:
param_grid_RF = {
    "n_estimators": [50, 100, 150],
    "max_depth": [3, 5, 7]
}

grid_RF = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid_RF,
    cv=5
)

grid_RF.fit(X_train_RF, y_train_RF)

print("Best Parameters :", grid_RF.best_params_)
print("Best Score :", round(grid_RF.best_score_, 4))

Best Parameters : {'max_depth': 7, 'n_estimators': 150}
Best Score : 0.9495


In [27]:
comparison = pd.DataFrame({
    "Model": [
        "SVM",
        "Random Forest",
        "AdaBoost",
        "Gradient Boosting",
        "XGBoost"
    ],
    "Accuracy": [
        accuracy_score(y_test, grid_search.best_estimator_.predict(X_test)),
        accuracy_RF,
        accuracy_Ada,
        accuracy_GB,
        accuracy_XGB
    ]
})

comparison["Accuracy"] = comparison["Accuracy"].round(4)

comparison = comparison.sort_values(
    by="Accuracy",
    ascending=False
)

print(comparison)

print("\nBest Model :")
print(comparison.iloc[0])

               Model  Accuracy
4            XGBoost    0.9734
1      Random Forest    0.9561
3  Gradient Boosting    0.9521
0                SVM    0.9415
2           AdaBoost    0.9415

Best Model :
Model       XGBoost
Accuracy     0.9734
Name: 4, dtype: object


In [28]:
import joblib

joblib.dump(modelXGB, "best_model.pkl")
joblib.dump(X.columns.tolist(), "columns.pkl")

print("Best Model Saved Successfully!")
print("Feature Columns Saved Successfully!")

Best Model Saved Successfully!
Feature Columns Saved Successfully!
